In [0]:
# Step 1: Read JSON file with multiline option
path = "/Volumes/dev_account/stage/account/raw/customer/inbound/Nested_json_data_corrected.jsonl"

# multiline=true: Reads entire JSON as one record (for pretty-printed JSON)
df = spark.read.option("multiline", "true").json(path)


## Understanding JSON Data Types in Spark

When Spark reads JSON, it infers the schema with these data types:

### 1. **Primitive Types**
* `string` - Text data
* `long` / `integer` - Numbers
* `double` - Decimal numbers
* `boolean` - true/false

### 2. **Complex Types**
* **`array`** - Collection of elements (like a list)
  - Example: `["item1", "item2", "item3"]`
  - Creates multiple rows when you **EXPLODE**
  
* **`struct`** - Nested object (like a dictionary)
  - Example: `{"city": "New York", "state": "NY"}`
  - Access fields with **dot notation** or **FLATTEN**

### Key Difference:
* **Array** = Multiple values → Use **EXPLODE** to create rows
* **Struct** = Nested fields → Use **dot notation** or **FLATTEN** to access

In [0]:
df.printSchema()

**Challenge:** How do we work with this nested data?

**Solutions:**
1. **EXPLODE** - Convert arrays to separate rows
2. **FLATTEN** - Access nested struct fields with dot notation

## Step 4: What is FLATTEN? 🔽

**FLATTEN** means accessing nested **struct** fields using **dot notation**.

### Before FLATTEN:
```
| id | address                                  |
|----|------------------------------------------|
| 1  | {city: "NYC", state: "NY", zip: 10001} |
```

### After FLATTEN:
```
| id | city | state | zip   |
|----|------|-------|-------|
| 1  | NYC  | NY    | 10001 |
```

### In PySpark:
```python
# Method 1: Dot notation
df.select("id", "address.city", "address.state")

# Method 2: Using col()
df.select(col("address.city"), col("address.state"))
```

**Use Case:** When you have nested structs and want to promote fields to top-level columns.

In [0]:
df3_explode=df2.select("attributes")

df4_explode=df3_explode.select(explode("attributes").alias("attributes"))

# display(df4_explode)

df_flattened=df4_explode.select("attributes.attributeId","attributes.name","attributes.property.addressdetails.city","attributes.property.addressdetails.state","attributes.property.addressdetails.country","attributes.property.addressdetails.postalCode")

# country:string
# postalCode:string
# state:string
# street:string

display(df_flattened)

In [0]:
# Step 4A: FLATTEN nested structs using dot notation
print("🔽 FLATTEN: Accessing nested struct fields\n")

# The root dataframe already has entityId, entityType, data, and relationships
# Let's select and display these top-level fields
df_entity_info = df.select(
    col("entityId").alias("entityId"),
    col("entityType").alias("entityType"),
    col("data").alias("data"),
    col("relationships").alias("relationships")
)

print("📊 Schema of entity fields:")
df_entity_info.printSchema()

print("\n📄 Entity Information:")
display(df_entity_info)

print("\n👉 Notice: Entity has nested 'data' struct and 'relationships' array!")

## Step 5: Combine EXPLODE + FLATTEN 🎯

For complex nested JSON, you often need to:
1. **EXPLODE** arrays to create rows
2. **FLATTEN** structs to create columns
3. Repeat for multiple levels of nesting

### Strategy:
```
Original Data
    ↓
EXPLODE array level 1
    ↓
FLATTEN struct fields
    ↓
EXPLODE nested array level 2
    ↓
FLATTEN nested struct fields
    ↓
Final flat table
```

In [0]:
# Step 5A: Complete example - Extract person information
print("🎯 COMPLETE EXAMPLE: Extract customer data\n")

# Step 1: Get entityId, entityType, and attributes array
df_step1 = df.select(
    col("entityId").alias("entityId"),
    col("entityType").alias("entityType"),
    col("data.attributes").alias("attributes")
)

print("✅ Step 1: Selected entity info and attributes")

# Step 2: Explode attributes array
df_step2 = df_step1.select(
    "entityId",
    "entityType",
    explode("attributes").alias("attribute")
)

print("✅ Step 2: Exploded attributes array")

# Step 3: Flatten attribute properties to get person details
df_final = df_step2.select(
    "entityId",
    "entityType",
    col("attribute.attributeId").alias("attributeId"),
    col("attribute.name").alias("attributeName"),
    col("attribute.property.firstName").alias("firstName"),
    col("attribute.property.lastName").alias("lastName"),
    col("attribute.property.age").alias("age"),
    col("attribute.property.contact.email").alias("email"),
    col("attribute.property.contact.phone").alias("phone"),
    col("attribute.property.addressDetails.street").alias("street"),
    col("attribute.property.addressDetails.city").alias("city"),
    col("attribute.property.addressDetails.state").alias("state"),
    col("attribute.property.addressDetails.postalCode").alias("postalCode")
)

print("✅ Transformation complete!")
print(f"📊 Rows: {df_final.count()}")
print("\n📊 Final Flattened Schema:")
df_final.printSchema()

print("\n📄 Final Flattened Data:")
display(df_final)

In [0]:
# Databricks / Spark supports 4 write modes when saving a table:
# 1. "overwrite" - Overwrites existing data in the table
# 2. "append"     - Adds new data to the existing table data
# 3. "ignore"     - Does nothing if the table already exists (silent skip)
# 4. "error"      - (default) Throws an error if the table already exists

# Example with each mode (uncomment the one you want to use):

# df_final.write.mode("overwrite").saveAsTable("dbacademy.stage.dim_account")
# df_final.write.mode("append").saveAsTable("dbacademy.stage.dim_account")
# df_final.write.mode("ignore").saveAsTable("dbacademy.stage.dim_account")
# df_final.write.mode("error").saveAsTable("dbacademy.stage.dim_account")

# Currently using default (error) mode:
df_final.write.saveAsTable("dbacademy.stage.dim_account")

In [0]:
%sql
select * from dbacademy.stage.dim_account

## 📚 Summary: Working with Nested JSON

### Key Concepts:

1. **JSON Schema Understanding**
   - Spark infers schema automatically
   - Two complex types: `array` and `struct`

2. **EXPLODE (Arrays → Rows)**
   ```python
   from pyspark.sql.functions import explode
   df.select(explode("array_column").alias("item"))
   ```
   - Converts array elements into separate rows
   - Use when: You have arrays and need each element as a row

3. **FLATTEN (Structs → Columns)**
   ```python
   df.select("struct_column.field1", "struct_column.field2")
   ```
   - Access nested fields with dot notation
   - Use when: You have nested objects and want flat columns

4. **Combine EXPLODE + FLATTEN**
   - For deeply nested JSON, chain operations:
     1. Explode outer array
     2. Flatten struct fields
     3. Explode inner arrays
     4. Flatten inner struct fields

### Common Pattern:
```python
df.select(explode("array1").alias("item")) \       # Explode array
  .select("item.field1",                      \       # Flatten struct
          "item.field2",                      \
          explode("item.nested_array").alias("nested")) \ # Explode nested
  .select("field1", "nested.nested_field")           # Flatten nested
```

### Next Steps:
- Practice with your own nested JSON files
- Try multiple levels of nesting
- Combine with SQL transformations

## 🔄 Understanding MERGE Statement in Databricks

### What is MERGE?
MERGE is a SQL statement that combines INSERT, UPDATE, and DELETE operations in a single transaction. It's also called "UPSERT" (UPDATE + INSERT).

### Why Use MERGE?
- **Slowly Changing Dimensions (SCD)**: Update existing records and insert new ones
- **Data Synchronization**: Keep target table in sync with source data
- **Incremental Loads**: Process only changed data efficiently
- **Idempotent Operations**: Can run same merge multiple times safely

### Basic MERGE Syntax:
```sql
MERGE INTO target_table
USING source_table
ON target.key = source.key
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT ...
WHEN NOT MATCHED BY SOURCE THEN DELETE
```

### Components:
1. **Target Table**: The table you want to update
2. **Source Data**: New/updated data (table, view, or subquery)
3. **Join Condition**: How to match source and target records
4. **Actions**: What to do when records match or don't match

In [0]:
tgt_schema=spark.table("dbacademy.stage.dim_account").schema
print(tgt_schema)

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType

df_select=df_final.select([
col(field.name).cast(field.dataType) for field in tgt_schema
if field.name in df_final.columns
])

In [0]:
# MERGE using DeltaTable API directly on DataFrame (no temp view needed)

spark.sql("delete from dbacademy.stage.dim_account")

from delta.tables import DeltaTable

# Build the target DeltaTable object
delta_table = DeltaTable.forName(spark, "dbacademy.stage.dim_account")

# Merge using the casted source DataFrame directly
# Match condition: composite key (entityId + attributeId)
delta_table.alias("target").merge(
    df_select.alias("source"),
    "target.entityId = source.entityId AND target.attributeId = source.attributeId"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


# Verify row count after merge
count_after = spark.table("dbacademy.stage.dim_account").count()
print(f"📊 Row count after MERGE: {count_after}")

## 🎯 MERGE Statement Components

### 1. **MERGE INTO** (Target Table)
```sql
MERGE INTO dbacademy.stage.dim_account AS target
```
- The table you want to update
- Must be a Delta table
- Alias it as 'target' for clarity

### 2. **USING** (Source Data)
```sql
USING source_data AS source
```
- Can be: table, view, or subquery
- Contains new/updated records
- Alias it as 'source' for clarity

### 3. **ON** (Match Condition)
```sql
ON target.entityId = source.entityId 
   AND target.attributeId = source.attributeId
```
- Defines how to match records
- Usually uses primary key or unique identifier
- Can have multiple conditions with AND/OR

### 4. **WHEN MATCHED** (Update Existing)
```sql
WHEN MATCHED THEN 
  UPDATE SET 
    target.firstName = source.firstName,
    target.lastName = source.lastName
```
- Executes when record EXISTS in both source and target
- Updates existing records
- Can add conditions: `WHEN MATCHED AND source.age > target.age`

### 5. **WHEN NOT MATCHED** (Insert New)
```sql
WHEN NOT MATCHED THEN
  INSERT (entityId, firstName, lastName, ...)
  VALUES (source.entityId, source.firstName, ...)
```
- Executes when record EXISTS in source but NOT in target
- Inserts new records
- Must specify all columns or use `INSERT *`

## 💡 MERGE Best Practices & Tips

### 1. **Match Condition (ON clause)**
```sql
-- ✅ GOOD: Use unique identifier(s)
ON target.id = source.id

-- ✅ GOOD: Composite key
ON target.entityId = source.entityId 
   AND target.attributeId = source.attributeId

-- ❌ BAD: Non-unique columns (creates duplicates)
ON target.city = source.city
```

### 2. **Data Type Matching**
- Always cast source DataFrame to match target schema
- Use `.cast()` in PySpark before creating temp view
- Prevents implicit conversion errors

### 3. **Performance Tips**
```sql
-- ✅ Optimize: Add WHERE clause to source
USING (
  SELECT * FROM source_data 
  WHERE last_modified > '2026-01-01'
) AS source

-- ✅ Optimize: Filter unnecessary updates
WHEN MATCHED AND source.value != target.value THEN UPDATE
```

### 4. **Testing Strategy**
1. Test with small dataset first
2. Use MERGE with INSERT only first
3. Then add UPDATE logic
4. Verify counts: `SELECT COUNT(*) FROM table`
5. Check for duplicates

### 5. **Common Patterns**

#### Pattern 1: Slowly Changing Dimension (SCD Type 1)
```sql
-- Overwrite old values with new values
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

#### Pattern 2: Incremental Load
```sql
-- Only insert new records, never update
WHEN NOT MATCHED THEN INSERT *
```

#### Pattern 3: Soft Delete
```sql
-- Mark as deleted instead of physical delete
WHEN MATCHED AND source.is_deleted = true THEN
  UPDATE SET target.is_active = false
```

### 6. **Error Handling**
```python
try:
    spark.sql("""
        MERGE INTO target USING source ON condition
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("✅ MERGE successful")
except Exception as e:
    print(f"❌ MERGE failed: {e}")
```

### 7. **Monitoring MERGE Results**
```sql
-- Check operation metrics
DESCRIBE HISTORY dbacademy.stage.dim_account LIMIT 1;

-- View last operation details
SELECT 
  operationMetrics.numTargetRowsInserted as inserted,
  operationMetrics.numTargetRowsUpdated as updated,
  operationMetrics.numTargetRowsDeleted as deleted
FROM (
  DESCRIBE HISTORY dbacademy.stage.dim_account LIMIT 1
);
```

In [0]:
%sql
-- Check the last MERGE operation metrics
DESCRIBE HISTORY dbacademy.stage.dim_account LIMIT 1;

## 📝 Summary: Complete MERGE Workflow

### Step-by-Step Checklist:

#### 1️⃣ **Preparation**
- ✅ Check target table schema (`DESCRIBE TABLE`)
- ✅ Check source DataFrame schema (`.printSchema()`)
- ✅ Cast source columns to match target types
- ✅ Create temporary view from DataFrame

#### 2️⃣ **Define Match Condition**
- ✅ Identify unique key columns (primary key)
- ✅ Write ON clause with proper join condition
- ✅ Test for duplicates in source data

#### 3️⃣ **Choose MERGE Pattern**
- **INSERT only**: New records only
  ```sql
  WHEN NOT MATCHED THEN INSERT *
  ```
  
- **UPDATE only**: Existing records only
  ```sql
  WHEN MATCHED THEN UPDATE SET *
  ```
  
- **UPSERT**: Both insert and update (most common)
  ```sql
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
  ```

#### 4️⃣ **Execute & Verify**
- ✅ Run MERGE statement
- ✅ Check row counts before/after
- ✅ Verify data with SELECT query
- ✅ Check DESCRIBE HISTORY for metrics

### Common MERGE Use Cases:

| Use Case | Pattern |
|----------|--------|
| Daily data sync | UPSERT (UPDATE + INSERT) |
| Append-only log | INSERT only |
| Dimension table update | UPSERT with conditions |
| Soft deletes | UPDATE with is_active flag |
| Incremental loads | INSERT with timestamp filter |

### Key Takeaways:
1. **MERGE = UPDATE + INSERT** in one statement
2. Always match data types between source and target
3. Use unique keys in ON clause
4. Test with small data first
5. Monitor operation metrics
6. MERGE works only on Delta tables

### Next Steps:
- Practice with your own datasets
- Implement SCD Type 2 (historical tracking)
- Try MERGE with complex conditions
- Optimize large MERGE operations